# Batch Inference Drift Monitoring

This notebook uses the Titanic train table as the reference population and the Titanic test table as the current daily batch. It scores the current batch with the production model, measures feature drift and model drift, and writes dashboard-ready monitoring tables to Unity Catalog.

In [0]:
import uuid
import warnings

import mlflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
mlflow.set_registry_uri("databricks-uc")

CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"

REFERENCE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.train"
CURRENT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.test"
MODEL_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model"
MODEL_ALIAS = "production"

PREDICTION_HISTORY_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.titanic_prediction_monitoring_history"
FEATURE_DRIFT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.feature_drift_daily"
MODEL_DRIFT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.model_drift_daily"
ALERT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.drift_alerts_daily"

PRIMARY_KEY = "PassengerId"
TARGET_COL = "Survived"
FEATURES = ["Fare", "Age", "Pclass", "SibSp", "Parch", "Sex"]
NUMERIC_FEATURES = ["Fare", "Age", "Pclass", "SibSp", "Parch"]
CATEGORICAL_FEATURES = ["Sex"]

BATCH_DATE = pd.Timestamp.utcnow().normalize().date()
BATCH_ID = f"batch_{pd.Timestamp.utcnow():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:8]}"

print(f"Reference data: {REFERENCE_TABLE}")
print(f"Current batch data: {CURRENT_TABLE}")
print(f"Production model: {MODEL_NAME}@{MODEL_ALIAS}")
print(f"Monitoring batch id: {BATCH_ID}")

In [0]:
reference_df = spark.table(REFERENCE_TABLE).toPandas()
current_df = spark.table(CURRENT_TABLE).toPandas()

reference_required_columns = [PRIMARY_KEY, TARGET_COL] + FEATURES
current_required_columns = [PRIMARY_KEY] + FEATURES

reference_missing_columns = sorted(set(reference_required_columns) - set(reference_df.columns))
current_missing_columns = sorted(set(current_required_columns) - set(current_df.columns))

if reference_missing_columns:
    raise ValueError(f"reference dataset is missing columns: {reference_missing_columns}")
if current_missing_columns:
    raise ValueError(f"current dataset is missing columns: {current_missing_columns}")

HAS_GROUND_TRUTH = TARGET_COL in current_df.columns
model_uri = f"models:/{MODEL_NAME}@{MODEL_ALIAS}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

reference_scored = reference_df[reference_required_columns].copy()
current_scored = current_df[current_required_columns].copy()
if HAS_GROUND_TRUTH:
    current_scored[TARGET_COL] = current_df[TARGET_COL]
else:
    current_scored[TARGET_COL] = np.nan

reference_scored["prediction"] = pd.Series(loaded_model.predict(reference_scored[FEATURES]), index=reference_scored.index)
current_scored["prediction"] = pd.Series(loaded_model.predict(current_scored[FEATURES]), index=current_scored.index)

reference_scored[TARGET_COL] = pd.to_numeric(reference_scored[TARGET_COL], errors="coerce")
current_scored[TARGET_COL] = pd.to_numeric(current_scored[TARGET_COL], errors="coerce")
reference_scored["prediction"] = pd.to_numeric(reference_scored["prediction"], errors="coerce")
current_scored["prediction"] = pd.to_numeric(current_scored["prediction"], errors="coerce")

monitoring_timestamp = pd.Timestamp.utcnow()
current_prediction_history = current_scored[[PRIMARY_KEY, TARGET_COL] + FEATURES + ["prediction"]].copy()
current_prediction_history["model_name"] = MODEL_NAME
current_prediction_history["model_alias"] = MODEL_ALIAS
current_prediction_history["batch_id"] = BATCH_ID
current_prediction_history["batch_date"] = BATCH_DATE
current_prediction_history["monitoring_timestamp"] = monitoring_timestamp
current_prediction_history = current_prediction_history.rename(columns={TARGET_COL: "actual"})

spark.createDataFrame(current_prediction_history).write.mode("append").saveAsTable(PREDICTION_HISTORY_TABLE)

print(f"Loaded {len(reference_scored):,} reference rows and {len(current_scored):,} current rows")
print(f"Ground truth available for current batch: {HAS_GROUND_TRUTH}")
print(f"Saved current batch predictions to {PREDICTION_HISTORY_TABLE}")
display(current_prediction_history.head(5))

In [0]:
EPSILON = 1e-6


def alert_from_score(score, moderate=0.10, high=0.20):
    if pd.isna(score):
        return "unknown"
    if score >= high:
        return "high"
    if score >= moderate:
        return "moderate"
    return "low"


def compute_psi(reference_series, current_series, bins=10):
    ref = pd.to_numeric(reference_series, errors="coerce").dropna()
    cur = pd.to_numeric(current_series, errors="coerce").dropna()

    if ref.empty or cur.empty:
        return np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    bin_edges = np.unique(ref.quantile(quantiles).values)

    if len(bin_edges) < 3:
        lower = min(ref.min(), cur.min())
        upper = max(ref.max(), cur.max())
        if lower == upper:
            return 0.0
        bin_edges = np.linspace(lower, upper, 4)

    ref_bins = pd.cut(ref, bins=bin_edges, include_lowest=True, duplicates="drop")
    cur_bins = pd.cut(cur, bins=bin_edges, include_lowest=True, duplicates="drop")

    ref_dist = ref_bins.value_counts(normalize=True, sort=False)
    cur_dist = cur_bins.value_counts(normalize=True, sort=False).reindex(ref_dist.index, fill_value=0.0)

    ref_values = np.clip(ref_dist.values, EPSILON, None)
    cur_values = np.clip(cur_dist.values, EPSILON, None)
    return float(np.sum((cur_values - ref_values) * np.log(cur_values / ref_values)))


def compute_tvd(reference_series, current_series):
    ref_dist = reference_series.fillna("__null__").astype(str).value_counts(normalize=True)
    cur_dist = current_series.fillna("__null__").astype(str).value_counts(normalize=True)

    categories = sorted(set(ref_dist.index).union(set(cur_dist.index)))
    ref_values = ref_dist.reindex(categories, fill_value=0.0).values
    cur_values = cur_dist.reindex(categories, fill_value=0.0).values
    return float(0.5 * np.abs(ref_values - cur_values).sum())


feature_drift_records = []

for feature_name in NUMERIC_FEATURES:
    ref_series = reference_scored[feature_name]
    cur_series = current_scored[feature_name]
    drift_score = compute_psi(ref_series, cur_series)

    feature_drift_records.append(
        {
            "batch_id": BATCH_ID,
            "batch_date": BATCH_DATE,
            "feature_name": feature_name,
            "feature_type": "numeric",
            "drift_metric": "population_stability_index",
            "drift_score": drift_score,
            "alert_level": alert_from_score(drift_score),
            "reference_mean": float(pd.to_numeric(ref_series, errors="coerce").mean()),
            "current_mean": float(pd.to_numeric(cur_series, errors="coerce").mean()),
            "reference_null_rate": float(ref_series.isna().mean()),
            "current_null_rate": float(cur_series.isna().mean()),
            "reference_top_value": None,
            "current_top_value": None,
        }
    )

for feature_name in CATEGORICAL_FEATURES:
    ref_series = reference_scored[feature_name]
    cur_series = current_scored[feature_name]
    drift_score = compute_tvd(ref_series, cur_series)

    reference_top_value = ref_series.fillna("__null__").astype(str).mode().iloc[0]
    current_top_value = cur_series.fillna("__null__").astype(str).mode().iloc[0]

    feature_drift_records.append(
        {
            "batch_id": BATCH_ID,
            "batch_date": BATCH_DATE,
            "feature_name": feature_name,
            "feature_type": "categorical",
            "drift_metric": "total_variation_distance",
            "drift_score": drift_score,
            "alert_level": alert_from_score(drift_score),
            "reference_mean": np.nan,
            "current_mean": np.nan,
            "reference_null_rate": float(ref_series.isna().mean()),
            "current_null_rate": float(cur_series.isna().mean()),
            "reference_top_value": reference_top_value,
            "current_top_value": current_top_value,
        }
    )

feature_drift_pdf = pd.DataFrame(feature_drift_records).sort_values("drift_score", ascending=False)
spark.createDataFrame(feature_drift_pdf).write.mode("append").saveAsTable(FEATURE_DRIFT_TABLE)

print(f"Saved feature drift metrics to {FEATURE_DRIFT_TABLE}")
display(feature_drift_pdf.head(10))

In [0]:
def safe_classification_metric(metric_name, y_true, y_pred):
    if metric_name == "accuracy":
        return float(accuracy_score(y_true, y_pred))
    if metric_name == "precision":
        return float(precision_score(y_true, y_pred, zero_division=0))
    if metric_name == "recall":
        return float(recall_score(y_true, y_pred, zero_division=0))
    if metric_name == "f1":
        return float(f1_score(y_true, y_pred, zero_division=0))
    raise ValueError(f"Unsupported metric: {metric_name}")


def compute_distribution_drift(reference_series, current_series):
    ref_dist = reference_series.fillna(-1).astype(int).value_counts(normalize=True)
    cur_dist = current_series.fillna(-1).astype(int).value_counts(normalize=True)
    labels = sorted(set(ref_dist.index).union(set(cur_dist.index)))
    ref_values = ref_dist.reindex(labels, fill_value=0.0).values
    cur_values = cur_dist.reindex(labels, fill_value=0.0).values
    return float(0.5 * np.abs(ref_values - cur_values).sum())


reference_pairs = reference_scored[[TARGET_COL, "prediction"]].dropna().astype(int).rename(columns={TARGET_COL: "actual"})
current_prediction_only = current_scored[["prediction"]].dropna().astype(int)

model_metric_thresholds = {
    "accuracy": 0.05,
    "precision": 0.05,
    "recall": 0.05,
    "f1": 0.05,
    "prediction_positive_rate": 0.10,
    "label_positive_rate": 0.10,
    "prediction_distribution_drift": 0.10,
}

model_drift_records = []
if HAS_GROUND_TRUTH:
    current_pairs = current_scored[[TARGET_COL, "prediction"]].dropna().astype(int).rename(columns={TARGET_COL: "actual"})

    for metric_name in ["accuracy", "precision", "recall", "f1"]:
        reference_value = safe_classification_metric(metric_name, reference_pairs["actual"], reference_pairs["prediction"])
        current_value = safe_classification_metric(metric_name, current_pairs["actual"], current_pairs["prediction"])
        drift_score = max(reference_value - current_value, 0.0)

        model_drift_records.append(
            {
                "batch_id": BATCH_ID,
                "batch_date": BATCH_DATE,
                "metric_name": metric_name,
                "reference_value": reference_value,
                "current_value": current_value,
                "delta": current_value - reference_value,
                "drift_score": drift_score,
                "alert_level": alert_from_score(
                    drift_score,
                    moderate=model_metric_thresholds[metric_name],
                    high=model_metric_thresholds[metric_name] * 2,
                ),
            }
        )

    metric_triplets = [
        ("prediction_positive_rate", float(reference_pairs["prediction"].mean()), float(current_pairs["prediction"].mean())),
        ("label_positive_rate", float(reference_pairs["actual"].mean()), float(current_pairs["actual"].mean())),
    ]
else:
    metric_triplets = [
        ("prediction_positive_rate", float(reference_pairs["prediction"].mean()), float(current_prediction_only["prediction"].mean())),
        (
            "prediction_distribution_drift",
            0.0,
            compute_distribution_drift(reference_pairs["prediction"], current_prediction_only["prediction"]),
        ),
    ]

for metric_name, reference_value, current_value in metric_triplets:
    drift_score = abs(current_value - reference_value)
    model_drift_records.append(
        {
            "batch_id": BATCH_ID,
            "batch_date": BATCH_DATE,
            "metric_name": metric_name,
            "reference_value": reference_value,
            "current_value": current_value,
            "delta": current_value - reference_value,
            "drift_score": drift_score,
            "alert_level": alert_from_score(
                drift_score,
                moderate=model_metric_thresholds[metric_name],
                high=model_metric_thresholds[metric_name] * 2,
            ),
        }
    )

if not HAS_GROUND_TRUTH:
    model_drift_records.append(
        {
            "batch_id": BATCH_ID,
            "batch_date": BATCH_DATE,
            "metric_name": "ground_truth_status",
            "reference_value": 1.0,
            "current_value": 0.0,
            "delta": -1.0,
            "drift_score": 0.0,
            "alert_level": "low",
            "note": "Current batch has no Survived label, so true performance drift cannot be computed until labels arrive.",
        }
    )

model_drift_pdf = pd.DataFrame(model_drift_records)
if "note" not in model_drift_pdf.columns:
    model_drift_pdf["note"] = None
model_drift_pdf = model_drift_pdf.sort_values(["alert_level", "drift_score"], ascending=[True, False])
spark.createDataFrame(model_drift_pdf).write.mode("append").saveAsTable(MODEL_DRIFT_TABLE)

print(f"Ground truth available for current batch: {HAS_GROUND_TRUTH}")
print(f"Saved model drift metrics to {MODEL_DRIFT_TABLE}")
display(model_drift_pdf)

In [0]:
feature_alerts = feature_drift_pdf.loc[
    feature_drift_pdf["alert_level"].isin(["moderate", "high"]),
    ["batch_id", "batch_date", "feature_name", "drift_metric", "drift_score", "alert_level"],
].copy()
feature_alerts["alert_type"] = "feature_drift"
feature_alerts["alert_message"] = feature_alerts.apply(
    lambda row: f"{row['feature_name']} drift score reached {row['drift_score']:.3f} using {row['drift_metric']}",
    axis=1,
)
feature_alerts = feature_alerts.rename(columns={"feature_name": "entity_name", "drift_metric": "metric_name", "drift_score": "metric_value"})

model_alerts = model_drift_pdf.loc[
    model_drift_pdf["alert_level"].isin(["moderate", "high"]),
    ["batch_id", "batch_date", "metric_name", "drift_score", "alert_level", "current_value", "reference_value"],
].copy()
model_alerts["alert_type"] = "model_drift"
model_alerts["alert_message"] = model_alerts.apply(
    lambda row: f"{row['metric_name']} moved from {row['reference_value']:.3f} to {row['current_value']:.3f}",
    axis=1,
)
model_alerts = model_alerts.rename(columns={"metric_name": "entity_name", "drift_score": "metric_value"})
model_alerts["metric_name"] = "metric_drift"
model_alerts = model_alerts[["batch_id", "batch_date", "entity_name", "metric_name", "metric_value", "alert_level", "alert_type", "alert_message"]]

alerts_pdf = pd.concat(
    [
        feature_alerts[["batch_id", "batch_date", "entity_name", "metric_name", "metric_value", "alert_level", "alert_type", "alert_message"]],
        model_alerts,
    ],
    ignore_index=True,
)

if alerts_pdf.empty:
    alerts_pdf = pd.DataFrame(
        [
            {
                "batch_id": BATCH_ID,
                "batch_date": BATCH_DATE,
                "entity_name": "batch_summary",
                "metric_name": "no_alerts",
                "metric_value": 0.0,
                "alert_level": "low",
                "alert_type": "summary",
                "alert_message": "No medium or high drift detected for this batch",
            }
        ]
    )

spark.createDataFrame(alerts_pdf).write.mode("append").saveAsTable(ALERT_TABLE)

print(f"Saved alerts to {ALERT_TABLE}")
display(alerts_pdf)

plt.figure(figsize=(10, 4))
plt.bar(feature_drift_pdf["feature_name"], feature_drift_pdf["drift_score"], color="#1f77b4")
plt.axhline(0.10, color="orange", linestyle="--", label="moderate threshold")
plt.axhline(0.20, color="red", linestyle="--", label="high threshold")
plt.title("Feature Drift Scores")
plt.ylabel("Drift score")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

metric_chart = model_drift_pdf[~model_drift_pdf["metric_name"].isin(["ground_truth_status"])].copy()
metric_chart["chart_label"] = metric_chart["metric_name"].str.replace("_", " ")

plt.figure(figsize=(10, 4))
positions = np.arange(len(metric_chart))
width = 0.35
plt.bar(positions - width / 2, metric_chart["reference_value"], width=width, label="reference")
plt.bar(positions + width / 2, metric_chart["current_value"], width=width, label="current")
plt.xticks(positions, metric_chart["chart_label"], rotation=20)
plt.ylim(0, 1)
plt.title("Reference vs Current Model Monitoring Metrics")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.show()